# Climate Comfort Index · XGBoost Regressor

**36120 Advanced Machine Learning Application** · Assessment Task 2 · Weather Intelligence (Open-Meteo, Sydney)

*Experiment notebook · predicts CCI +1, +2 and +3 days ahead*

This notebook trains and evaluates the **XGBoost Regressor** model for the **Climate Comfort Index (CCI)**, using the leakage-audited feature matrix produced by the preprocessing notebook.

### Student details

| | |
|:--|:--|
| **Student name** | Juan Vargas T. |
| **Student ID** | 25945320 |
| **Subject** | 36120 Advanced Machine Learning Application |
| **Assessment** | AT2 · Weather Intelligence |
| **GitHub** | [Sebas102507](https://github.com/Sebas102507) |

## Overview

### Hypothesis

Gradient-boosted trees will predict CCI **more accurately than the Random Forest and both
baselines at all three horizons**, because boosting fits residuals sequentially and can exploit
weak signal that bagging averages away.

The competing possibility: with about 2,900 training rows and weak persistence at *t*+3, boosting
may overfit where bagging does not. The generalisation check (section 10) and seed stability
(section 7) are the evidence either way.

### Design

| Choice | Value | Reason |
|:--|:--|:--|
| Algorithm | `XGBRegressor`, `hist` tree method | Strong tabular performance; fast on a dataset this size |
| Scaling | None | Trees split on thresholds, so scale is irrelevant |
| Strategy | One model per horizon (the endpoint returns three values) | Matches the deployed artefact layout |
| Selection | Optuna TPE sampler, 30 trials per target | See section 6 |
| Objective | validation **mean absolute error (MAE)**, in CCI points (lower is better) | See section 6 |

### Workflow

All ten experiment notebooks follow the same steps and run the same code from
`nucleus_ml.weather.Experiment`, so their results are directly comparable. **Only the model
specification in section 1 differs between notebooks.**

| # | Step | Purpose |
|:-:|:--|:--|
| 1 | Model specification | The only algorithm-specific cell |
| 2 | Data | Training and validation splits (the test split is never loaded here) |
| 3 | Baselines | The floor every model must clear |
| 4 | Default configuration | An untuned reference point |
| 5 | Feature-family ablation | What each block of engineered features contributes |
| 6 | Hyperparameter selection | Choose settings on the validation split |
| 7 | Seed stability | Is the tuning gain real or noise? |
| 8 | Final model | Tuned model against the default and the baselines |
| 9 | Feature importance | Which features the model relies on |
| 10 | Generalisation check | Training vs validation performance |
| 11 | Save artefacts | Model bundle + `/model-metadata` document |
| 12 | Conclusions | What to carry into the report |

---
## 0. Setup

Imports, the experiment's identity and the task definition. Everything else comes from the
project package, so no helper is redefined in this notebook.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from xgboost import XGBRegressor

from nucleus_ml import weather as wx

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

TASK = "comfort_climate"
EXPERIMENT = "cci_xgboost"
ALGORITHM = "XGBoost Regressor"
N_TRIALS = 30  # Optuna trials per target (None when a fixed grid is used)

task = wx.load_task(TASK)
print(f"{task.target_name} | {ALGORITHM} | {len(task.feature_columns)} features | "
      f"targets: {task.target_columns}")

---
## 1. Model specification

This is the **only algorithm-specific cell**. It defines how the estimator is built, a sensible
default configuration, a cheaper configuration for the ablation, and the candidate settings for
section 6 (either an Optuna search space or a fixed grid).

`max_depth` and `min_child_weight` control tree complexity and are the main defence against
overfitting. `subsample` and `colsample_bytree` add randomness, which helps with the blocks of
near-duplicate columns. `n_estimators` is tuned rather than set by early stopping, because early
stopping would use the validation split as a fitting signal on every trial.

In [ ]:
def build_estimator(params: dict, seed: int) -> XGBRegressor:
    """Gradient-boosted regression trees."""
    return XGBRegressor(tree_method="hist", n_jobs=-1, random_state=seed, verbosity=0, **params)


DEFAULT_PARAMS = {"n_estimators": 500, "learning_rate": 0.04, "max_depth": 4,
                  "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 0.6,
                  "reg_lambda": 1.0, "reg_alpha": 0.0}
ABLATION_PARAMS = {**DEFAULT_PARAMS, "n_estimators": 150, "learning_rate": 0.08}


def search_space(trial) -> dict:
    """Boosting rounds, learning rate, tree complexity, subsampling and regularisation."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 150, 900, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 7),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 30),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 50.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
    }


SEARCH_SPACE, GRID = search_space, None

In [ ]:
exp = wx.Experiment(
    task,
    name=EXPERIMENT,
    algorithm=ALGORITHM,
    build=build_estimator,
    default_params=DEFAULT_PARAMS,
    ablation_params=ABLATION_PARAMS,
    search_space=SEARCH_SPACE,
    grid=GRID,
)

---
## 2. Data

Only the **training** and **validation** splits are loaded. The test split is reserved for a
single evaluation in the comparison notebook, once one model has been selected, so nothing in
this notebook can be influenced by it.

> The preprocessing notebook already applied the chronological split, the warm-up filter and an
> embargo at each boundary. Nothing is re-split or shuffled here.

In [ ]:
exp.data_summary()

---
## 3. Baselines

Two naive forecasts, both fitted on the training split only:

| Baseline | Rule | What it knows |
|:--|:--|:--|
| **Persistence** | Tomorrow's CCI equals today's CCI | The current weather, but not the season |
| **Climatology** | The average CCI for that day of the year | The season, but not the current weather |

A model must beat **both** to be worth deploying: that shows it has combined current conditions
with the seasonal cycle.

In [ ]:
exp.run_baselines()

---
## 4. Default configuration

The untuned default is scored before any search. It is the reference for judging what tuning
bought in section 8; a search that fails to beat a sensible default is a result worth reporting.
`best_baseline` is repeated for comparison.

In [ ]:
exp.run_defaults()

---
## 5. Feature-family ablation

Feature families are added **one at a time**, starting from persistence alone (a single column),
and the model is refitted at each step. This measures what each block of engineering contributes
instead of assuming it. The cheaper `ABLATION_PARAMS` are used throughout so the differences come
from the features, not from tuning. Scores are on validation (MAE).

In [ ]:
ablation = exp.run_ablation()
plt.show()
ablation

---
## 6. Hyperparameter selection

The objective is validation **mean absolute error (MAE)**, in CCI points (lower is better).

**Why MAE.** It is measured in CCI points, so an improvement of 0.4 reads directly as "0.4 comfort
points closer on average". RMSE squares errors and would be dominated by the rare heavy-rain days
where CCI collapses, tuning the model for the tail instead of the ordinary days that make up
almost all API traffic.

**Why one search per horizon.** Predictability decays with lead time, so a single averaged score
would let a strong *t*+1 model hide a weak *t*+3 one.

**Why validation, not cross-validation.** The split is chronological. Shuffled k-fold would put
future days into training and inflate every score, because consecutive days are strongly
correlated. The fixed 2024 validation year matches how the model will be used: trained on the
past, applied forward.

> **Stated plainly:** selecting on validation makes the validation score slightly optimistic. The
> honest estimate comes from the test split, which only the comparison notebook reads, once.

In [ ]:
exp.tune(n_trials=N_TRIALS)

In [ ]:
exp.plot_search()
plt.show()

---
## 7. Seed stability

The selected configuration is refitted with three different random seeds. If the score moves by
as much as the spread between the top search candidates (`search_beats_noise = False`), the search
was ranking noise and the conclusion should say so. Deterministic models show a spread of zero.

In [ ]:
exp.seed_stability()

---
## 8. Final model against the baselines

The selected configuration is refitted and compared with the default configuration and every
baseline. `gain_of_tuned_model` is positive when the tuned model is better than that row.

`pred_std` is shown next to the error metrics. A spread far below the observed spread means the
model is playing safe by predicting close to the average, which can give a respectable MAE while
missing exactly the uncomfortable days users care about. Expect error to grow with the horizon.

In [ ]:
exp.finalise()

In [ ]:
exp.plot_validation()
plt.show()

---
## 9. Feature importance

Tree ensembles report **impurity-based importance** directly from the fitted model. Comparing horizons is informative: features that rise in rank from *t*+1 to *t*+3 carry genuine forecast signal rather than persistence.

> Correlated features (a variable, its lags and its rolling means) share importance, so a feature
> that looks unimportant on its own may belong to an essential block. The ablation in section 5 is
> the better guide to what can be removed.

In [ ]:
family_importance = exp.importance()
plt.show()
family_importance

---
## 10. Generalisation check

Training and validation scores side by side. A small gap means the model generalises; a large
gap means it has memorised patterns in the training years that do not carry over to 2024. Read
it together with the seed-stability table in section 7.

In [ ]:
exp.generalisation()

---
## 11. Save artefacts

The fitted model(s) are written to `models/comfort_climate/experiments/cci_xgboost/` with:

| File | Contents |
|:--|:--|
| `cci_xgboost_h<horizon>.joblib` | One fitted model per horizon |
| `metadata.json` | Target, algorithm, horizon, features, importance, hyperparameters, metrics — the `/model-metadata` document |
| `manifest.json` | File list and the feature column order the API must use |

> The feature column **order** is recorded because models receive an unnamed array at prediction
> time and would silently give wrong answers if the columns were permuted.

In [ ]:
exp.export()

---
## 12. Conclusions

- **Against the baselines.** The section 8 table is the headline. A model that cannot beat
  persistence at *t*+1 has a feature problem, not a tuning problem.
- **What tuning bought.** Compare `tuned` with `default`. If the gain is smaller than the seed
  spread in section 7, report the search as inconclusive.
- **Regularisation across horizons.** If `reg_lambda` rises or `max_depth` falls as the horizon
  lengthens, that is direct evidence of a weakening signal at longer lead times.

> The test split has still not been used. It is scored exactly once, in the
> `cci_comparison` notebook, after one model has been selected on validation
> performance across all five CCI experiments.